# Discriminant Classifiers

Using discriminant classifiers / Naive Bayes to classify our data.

In [1]:
import pandas as pd

In [2]:
learn_data = pd.read_csv("preprocess_train_v3.csv", header = None)
learn_data.columns = ['Age', 'ALB', 'AR', 'IBRatio', 'AST_ALT_Ratio', 'LogIB', 'LogAlkphos', 'LogSgpt', 'LogSgot', 'Female', 'Target']
learn_data["Female"] = learn_data["Female"].astype("category")
learn_data["Target"] = learn_data["Target"].astype("category")
learn_data.head()

,Age,ALB,AR,IBRatio,AST_ALT_Ratio,LogIB,LogAlkphos,LogSgpt,LogSgot,Female,Target
0,48,2.4,0.52,0.488889,5.692308,7.884574e-01,5.641907,2.564949,4.304065,0,0
1,39,4.3,1.38,0.526316,1.476190,-1.110223e-16,5.192957,3.737670,4.127134,0,0
2,23,3.1,1.00,0.700000,1.951220,-3.566749e-01,5.356586,3.713572,4.382027,0,0
3,42,3.2,1.06,0.714286,2.314286,-6.931472e-01,5.023881,3.555348,4.394449,1,0
4,54,3.4,0.80,0.495575,1.233333,2.415914e+00,6.324359,3.401197,3.610918,1,0


In [3]:
learn_data.isna().value_counts()

Age    ALB    AR     IBRatio  AST_ALT_Ratio  LogIB  LogAlkphos  LogSgpt  LogSgot  Female  Target
False  False  False  False    False          False  False       False    False    False   False     451
Name: count, dtype: int64

In [4]:
X = learn_data.drop(columns = ["Target", "Female"])
y = learn_data["Target"]

## Metrics

In [5]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall, prec, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

metrics_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

## Linear Discriminant Classifier

From PCA analysis, we know that our data can be separated in two clouds of sick and healthy patients respectively. A LD classifier might work well, but we know that the clouds may overlap. Also, their covariance matrices are clearly different. We might need to use a Quadratic Discriminant Classifier instead.

In [6]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.33, random_state = 42)

lda_model = LinearDiscriminantAnalysis(priors = (0.5, 0.5))
lda_model.fit(X_train, y_train)

print('Priors:', lda_model.priors_)

Priors: [0.5 0.5]


In [7]:
confusion(np.array(y_train), pd.Series(lda_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	62	15
	0	92	133
Accuracy: 64.57%


In [8]:
confusion(np.array(y_val), pd.Series(lda_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	43	9
	0	43	54
Accuracy: 65.10%


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

LDA_pipeline = Pipeline([('scaler', StandardScaler()), ('LDA', LinearDiscriminantAnalysis())])

n = 10
priors = [(p / n, 1 - p / n) for p in range(1, n)]

LDA_search = GridSearchCV(estimator = LDA_pipeline,
                          param_grid = {'LDA__priors' : priors},
                          scoring = 'f1_macro',
                          cv = 5)
LDA_search.fit(X, y)
LDA_search.best_params_

{'LDA__priors': (0.5, 0.5)}

In [10]:
from sklearn.model_selection import cross_validate

lda_model = LinearDiscriminantAnalysis(priors = (0.6, 0.4))
lda_pipeline = Pipeline([('scaler', StandardScaler()), ('LDA', lda_model)])

cross_val_results = pd.DataFrame(cross_validate(lda_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["LDA", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LDA,0.634796,0.648101,0.633869,0.682955


## Quadratic Discrimant Classifier

We now use a QDA classifier. We see that the problem is preserved: the "sick" class overlaps too much with the healthy class and it

In [11]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

qda_model = QuadraticDiscriminantAnalysis(priors = (0.5, 0.5))
_ = qda_model.fit(X_train, y_train)

In [12]:
confusion(np.array(y_train), pd.Series(qda_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	68	9
	0	100	125
Accuracy: 63.91%


In [13]:
confusion(np.array(y_val), pd.Series(qda_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	43	9
	0	45	52
Accuracy: 63.76%


QDA can be regularized with a parameter between 0 and 1, so we can apply cross-validation in an attempt to obtain better metrics. In general, a small value of this regularization parameter (between 0.01 and 0.1) is desirable, but it does not improve results by much.

In [14]:
QDA_pipeline = Pipeline([('scaler', StandardScaler()), ('QDA', QuadraticDiscriminantAnalysis(priors = (0.5, 0.5)))])

n = 10
regs = np.logspace(start = -4, stop = -0.5, num = 100)

QDA_search = GridSearchCV(estimator = QDA_pipeline,
                          param_grid = {'QDA__reg_param' : regs},
                          scoring = 'f1_macro',
                          cv = 5)
QDA_search.fit(X, y)
QDA_search.best_params_

{'QDA__reg_param': 0.0042292428743894986}

In [15]:
QDA_search.best_score_

0.6380406774077192

In [16]:
QDA_reg_param_ = QDA_search.best_params_['QDA__reg_param']
qda_model = QuadraticDiscriminantAnalysis(priors = (0.5, 0.5),
                                          reg_param = QDA_reg_param_)
qda_pipeline = Pipeline([('scaler', StandardScaler()), ('QDA', qda_model)])

cross_val_results = pd.DataFrame(cross_validate(qda_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["QDA", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
QDA,0.638041,0.709058,0.673465,0.647424
LDA,0.634796,0.648101,0.633869,0.682955


## Naive Bayes



In [17]:
from sklearn.naive_bayes import GaussianNB

gaussian_nb = GaussianNB(priors = (0.5, 0.5))
gaussian_nb.fit(X_train, y_train)

confusion(np.array(y_train), pd.Series(gaussian_nb.predict(X_train)))

		Predicted
		+1	0
Real	+1	66	11
	0	105	120
Accuracy: 61.59%


In [18]:
confusion(np.array(y_val), pd.Series(gaussian_nb.predict(X_val)))

		Predicted
		+1	0
Real	+1	46	6
	0	46	51
Accuracy: 65.10%


In [19]:
gaussian_nb = GaussianNB(priors = (0.5, 0.5))
NB_pipeline = Pipeline([('scaler', StandardScaler()), ('NB', gaussian_nb)])

cross_val_results = pd.DataFrame(cross_validate(NB_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["GaussianNB", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
QDA,0.638041,0.709058,0.673465,0.647424
LDA,0.634796,0.648101,0.633869,0.682955
GaussianNB,0.617506,0.690962,0.659558,0.625299


## Logistic Regression

In [20]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV

logreg_model = LogisticRegression(C = 20, random_state = 42, class_weight = "balanced")

logreg_model.fit(X_train, y_train)
confusion(np.array(y_train), pd.Series(logreg_model.predict(X_train)))

		Predicted
		+1	0
Real	+1	64	13
	0	89	136
Accuracy: 66.23%


/home/simple/.local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [21]:
confusion(np.array(y_val), pd.Series(logreg_model.predict(X_val)))

		Predicted
		+1	0
Real	+1	43	9
	0	40	57
Accuracy: 67.11%


In [22]:
logreg_pipeline = Pipeline([('scaler', StandardScaler()), ('logreg', LogisticRegression(class_weight = "balanced"))])

n = 100
m = 10
Cs = np.logspace(start = -4, stop = 2, num = n)

logreg_search = GridSearchCV(estimator = logreg_pipeline,
                             param_grid = {'logreg__C' : Cs},
                             scoring = 'f1_macro',
                             cv = 5)
logreg_search.fit(X, y)
logreg_search.best_params_

{'logreg__C': 0.10722672220103231}

In [23]:
logreg_search.best_score_

0.6561321598937985

In [24]:
logreg_C = logreg_search.best_params_["logreg__C"]

logreg_model_best = LogisticRegression(C = logreg_C,
                                       class_weight = "balanced")
logreg_pipeline = Pipeline([('scaler', StandardScaler()), ('logreg', logreg_model_best)])

cross_val_results = pd.DataFrame(cross_validate(logreg_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["LogReg-Best", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LogReg-Best,0.656132,0.715346,0.67638,0.66967
QDA,0.638041,0.709058,0.673465,0.647424
LDA,0.634796,0.648101,0.633869,0.682955
GaussianNB,0.617506,0.690962,0.659558,0.625299


## Trying our best classifiers on our test data

In [25]:
test_data = pd.read_csv("preprocess_test_v3.csv", header = None)
test_data.columns = ['Age', 'ALB', 'AR', 'IBRatio', 'AST_ALT_Ratio', 'LogIB', 'LogAlkphos', 'LogSgpt', 'LogSgot', 'Female']
test_data = test_data.drop(columns = "Female")
test_data.head()

,Age,ALB,AR,IBRatio,AST_ALT_Ratio,LogIB,LogAlkphos,LogSgpt,LogSgot
0,11,4.2,1.40,0.857143,1.115385,-0.510826,6.383507,3.258097,3.367296
1,62,4.0,0.80,0.500000,2.246377,-0.105361,5.411646,4.234107,5.043425
2,60,4.2,1.10,0.714286,0.437500,-0.693147,5.159055,3.465736,2.639057
3,60,3.2,0.78,0.508772,2.063107,1.064711,5.365976,6.021023,6.745236
4,48,2.7,0.90,0.777778,2.250000,-0.356675,5.164786,3.178054,3.988984


In [26]:
test_y = pd.read_csv("test_y.csv").iloc[:, 1]
test_y

0      1
1      0
2      1
3      0
4      1
      ..
111    1
112    0
113    1
114    0
115    0
Name: Label, Length: 116, dtype: int64

### LDA

In [27]:
lda_pipeline.fit(X, y)

labels_lda = pd.DataFrame(columns = ['ID', 'Label'])
labels_lda['Label'] = pd.DataFrame(lda_pipeline.predict(test_data))
labels_lda['ID'] = labels_lda.index + 1
labels_lda.to_csv('new_predictions/lda_best_fs.csv', index = False)
labels_lda

,ID,Label
0,1,1
1,2,0
2,3,0
3,4,0
4,5,1
...,...,...
111,112,0
112,113,0
113,114,1
114,115,0


In [28]:
compute_metrics(test_y, labels_lda['Label'])

[0.6576388888888889,
 0.6673968601679445,
 0.6526806526806527,
 0.7068965517241379]

### QDA

In [29]:
qda_pipeline.fit(X, y)

labels_qda = pd.DataFrame(columns = ['ID', 'Label'])
labels_qda['Label'] = pd.DataFrame(qda_pipeline.predict(test_data))
labels_qda['ID'] = labels_qda.index + 1
labels_qda.to_csv('new_predictions/qda_best_fs.csv', index = False)
labels_qda

,ID,Label
0,1,0
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,1
112,113,0
113,114,1
114,115,0


In [30]:
compute_metrics(test_y, labels_qda['Label'])

[0.644535240040858, 0.6920408908360716, 0.656547619047619, 0.6637931034482759]

### Naive Bayes

In [31]:
NB_pipeline.fit(X, y)

labels_nb = pd.DataFrame(columns = ['ID', 'Label'])
labels_nb['Label'] = pd.DataFrame(NB_pipeline.predict(test_data))
labels_nb['ID'] = labels_nb.index + 1
labels_nb.to_csv('new_predictions/nb_best_fs.csv', index = False)
labels_nb

,ID,Label
0,1,0
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,1
112,113,0
113,114,1
114,115,0


In [32]:
compute_metrics(test_y, labels_nb['Label'])

[0.6577639751552795,
 0.7163198247535597,
 0.6761819803746655,
 0.6724137931034483]

### Logistic Regression

In [33]:
logreg_pipeline.fit(X, y)

labels_logreg = pd.DataFrame(columns = ['ID', 'Label'])
labels_logreg['Label'] = pd.DataFrame(logreg_pipeline.predict(test_data))
labels_logreg['ID'] = labels_logreg.index + 1
labels_logreg.to_csv('new_predictions/logreg_best_fs.csv', index = False)
labels_logreg

,ID,Label
0,1,1
1,2,0
2,3,1
3,4,0
4,5,1
...,...,...
111,112,1
112,113,0
113,114,1
114,115,1


In [34]:
compute_metrics(test_y, labels_logreg['Label'])

[0.6992221261884184,
 0.7555677254472435,
 0.7083333333333333,
 0.7155172413793104]

### Discrepancies

In [35]:
np.logical_and(labels_qda == labels_lda, labels_qda == labels_logreg).value_counts()

ID    Label
True  True     89
      False    27
Name: count, dtype: int64

In [36]:
(labels_qda == labels_logreg).value_counts()

ID    Label
True  True     102
      False     14
Name: count, dtype: int64